# SIFT — Tier A on the full 2000-probe manifest

Runs **only SIFT**, on the SAME 2000-probe manifest as Gabor/DINOv2 (overriding
its nested-500 cap with `--no-cap`), all 3 levels × 2 conditions.

**No GPU needed** — SIFT is OpenCV on CPU (keypoints + FLANN + RANSAC). Use a
**CPU** session; the GPU would sit idle.

**Time (measured ~3.6 s/probe with 4 workers):** ~2 h per condition, ~4 h per
level, ~12 h for all three. So **commit ONE level at a time** (set `LEVELS`
below): each commit is ~4 h, safely under the limit and resume-safe.

**Tier B (full probe set) is NOT included for SIFT:** it is O(N×gallery), ~17 h
*per condition* (≈18000 probes × 6000 gallery), i.e. days in total — infeasible.
SIFT's 2000-probe Tier A is its best feasible estimate.

Use **Save Version → "Save & Run All (Commit)"** so a closed tab can't lose the
output; download `results/` from the committed version.

In [ ]:
import os
if not os.path.exists('/kaggle/working/Biometric'):
    !git clone https://github.com/AndreaGiurissich/Biometric.git /kaggle/working/Biometric
%cd /kaggle/working/Biometric
!git pull origin main
!pip install -q -r requirements.txt
!python scripts/verify_dataset.py --input-root /kaggle/input

## Run SIFT (Tier A, 2000 probes)

Edit `LEVELS` to split across commits — e.g. `['Easy']` for the first commit,
then `['Medium']`, then `['Hard']`. Each level does both conditions (~4 h).

In [ ]:
import subprocess
LEVELS = ['Easy', 'Medium', 'Hard']   # <- set to one level per commit if needed
for lv in LEVELS:
    print(f'\n===== SIFT {lv} (Tier A, 2000) =====')
    subprocess.run(['python', 'scripts/run_model.py', '--model', 'sift',
                    '--no-cap', '--level', lv, '--condition', 'both',
                    '--workers', '4'], check=True)

## SIFT analysis + save

Tables/figures/significance for whatever SIFT results exist in this session. To
compare against Gabor/DINOv2, merge this `results/raw/` with theirs and run
`synthesize.py`/`significance.py --results-dir <merged>` (e.g. locally).

In [ ]:
!python scripts/synthesize.py
!python scripts/significance.py
!python scripts/save_results.py --name sift_results